# Week 9 Walkthrough — From a CSV to a Report

**Topic:** Reading and writing files, the with statement, CSV, error handling

Data lives in files. We'll write one, read it back, handle the case where it's missing, and turn it into a printed report — the full round trip, with the `csv` module doing the fiddly parts.

Run the cells in order. Each step adds one idea to the program, and the last
section pulls the whole thing together. Change things and re-run — that is the
whole point of a notebook.

## Step 1 — Writing a text file


`with open(...)` opens the file and closes it for you, even if something goes
wrong in between. Always use it.

In [ ]:
with open("notes.txt", "w") as file:
    file.write("First line\n")
    file.write("Second line\n")

print("written")

## Step 2 — Reading it back


Three ways: all at once, line by line as a list, or by iterating. The third is
usually best — it doesn't load the whole file into memory.

In [ ]:
with open("notes.txt", "r") as file:
    print(repr(file.read()))

with open("notes.txt", "r") as file:
    for number, line in enumerate(file, start=1):
        print(number, line.strip())

## Step 3 — Modes matter — 'w' destroys


`'w'` truncates the file to nothing before writing. `'a'` appends. Getting this
wrong is how people lose data.

In [ ]:
with open("notes.txt", "a") as file:
    file.write("Third line, appended\n")

with open("notes.txt") as file:
    print(file.read())

## Step 4 — Write a CSV properly


You *could* join values with commas yourself. Don't — the moment a title
contains a comma, your file is broken. The `csv` module handles quoting.

In [ ]:
import csv

rows = [
    ["title", "author", "loans"],
    ["Dune", "Herbert, Frank", 42],
    ["Beloved", "Morrison, Toni", 93],
    ["Neuromancer", "Gibson, William", 17],
    ["Silent Spring", "Carson, Rachel", 61],
]

with open("circulation.csv", "w", newline="") as file:
    csv.writer(file).writerows(rows)

with open("circulation.csv") as file:
    print(file.read())

## Step 5 — Read a CSV back


`csv.reader` hands you a list per row. Remember the header row is just another
row — skip it with `next()`.

In [ ]:
import csv

with open("circulation.csv") as file:
    reader = csv.reader(file)
    header = next(reader)
    print("columns:", header)
    for row in reader:
        print(row)

## Step 6 — DictReader is usually nicer


It uses the header row to give you a dictionary per record, so you refer to
columns by name instead of counting positions.

In [ ]:
import csv

with open("circulation.csv") as file:
    for record in csv.DictReader(file):
        print(f"{record['title']:16} {record['loans']:>4}")

## Step 7 — Everything from a file is a string


`record['loans']` is `'42'`, not `42`. Convert before you calculate.

In [ ]:
import csv

with open("circulation.csv") as file:
    records = list(csv.DictReader(file))

print(type(records[0]["loans"]))
total = sum(int(r["loans"]) for r in records)
print("total loans:", total)

## Step 8 — Handle the file not being there


Files go missing, get renamed, sit in the wrong folder. Say what should happen
instead of letting the program die.

In [ ]:
import csv

def load(path):
    try:
        with open(path, newline="") as file:
            return list(csv.DictReader(file))
    except FileNotFoundError:
        print(f"No file at {path} - starting with nothing")
        return []
    except PermissionError:
        print(f"Not allowed to read {path}")
        return []

print(len(load("circulation.csv")), "records")
print(len(load("does-not-exist.csv")), "records")

---

## The finished program

Everything above, in one place. This is the version worth keeping.


Read the data, compute, write a report file, then show it. This is the shape of
an enormous amount of real work.

In [ ]:
# Week 9 - CSV in, report out

import csv

SOURCE = "circulation.csv"
REPORT = "circulation_report.txt"


def load(path):
    try:
        with open(path, newline="") as file:
            return [
                {"title": r["title"], "author": r["author"], "loans": int(r["loans"])}
                for r in csv.DictReader(file)
            ]
    except FileNotFoundError:
        print(f"Could not find {path}")
        return []
    except (KeyError, ValueError) as e:
        print(f"{path} is not shaped the way we expect: {e}")
        return []


def write_report(records, path):
    ranked = sorted(records, key=lambda r: r["loans"], reverse=True)
    total = sum(r["loans"] for r in records)

    with open(path, "w") as file:
        file.write("CIRCULATION REPORT\n")
        file.write("=" * 46 + "\n\n")
        for place, r in enumerate(ranked, start=1):
            file.write(f"{place}. {r['title']:22} {r['loans']:4}\n")
        file.write("\n" + "-" * 46 + "\n")
        file.write(f"Titles:  {len(records)}\n")
        file.write(f"Loans:   {total}\n")
        file.write(f"Average: {total / len(records):.1f}\n")


records = load(SOURCE)

if records:
    write_report(records, REPORT)
    with open(REPORT) as file:
        print(file.read())
else:
    print("Nothing to report.")

---

## Try it yourself

Use the empty cells below. There is no grade attached — this is where the
learning actually happens.

**1.** Add a row to `circulation.csv` with a title containing a comma. Read it back with `csv.DictReader` — the module handles the quoting for you.

**2.** Change `load` to skip any row where `loans` isn't a number, instead of giving up on the whole file.

**3.** Write the report as a CSV instead of plain text, with a `rank` column.

**4.** Open `circulation_report.txt` in a text editor to confirm it really is on disk.

In [ ]:
# Try it yourself 1

In [ ]:
# Try it yourself 2

In [ ]:
# Try it yourself 3